# Wake-domain topo editor

Interactive `mom6_forge.topo_editor.TopoEditor` on a flat 144x100 grid (same size/resolution/location as the `wake_domain` case), so whatever mask you draw here drops straight back into `wake_case.py`.

Reference images in this folder:
- `reference_sketch.png` -- the original hand-drawn 7-segment sketch this domain is based on.
- `current_topo.png` -- my best attempt at it in code, for comparison (2 headlands, 2 corner squares, 1 patch, 7 OBC segments overlaid).

Use the **mom6_forge** conda env / Jupyter kernel -- that's where `ipywidgets`/`cartopy`/`topo_editor` are installed.

Editor controls: toggle "Rectangular Area" to drag-select a region, then use the **Mask** buttons (Land/Ocean) to paint it -- much faster than double-clicking cell by cell. Undo/Redo work since `git=True` below.

In [ ]:
from pathlib import Path
from mom6_forge.grid import Grid
from mom6_forge.topo import Topo
from mom6_forge.topo_editor import TopoEditor

NX = 144
NY = 100
RESOLUTION = 1.0 / 12.0
XSTART = -45.0
YSTART = 32.0

grid = Grid(
    lenx=NX * RESOLUTION,
    leny=NY * RESOLUTION + 1e-6,  # avoid float floor-rounding grid.ny down by 1
    resolution=RESOLUTION,
    xstart=XSTART,
    ystart=YSTART,
    name="wake_domain",
)
assert grid.nx == NX and grid.ny == NY, f"grid is {grid.nx}x{grid.ny}, expected {NX}x{NY}"
grid.nx, grid.ny

In [ ]:
VERSION_CONTROL_DIR = Path("TopoLibrary").resolve()

topo = Topo(grid, min_depth=10.0, version_control_dir=VERSION_CONTROL_DIR, git=True)
topo.set_flat(1000.0)

In [ ]:
editor = TopoEditor(topo)
editor

## When you're happy with the mask

Run the cell below to dump a plain numpy boolean `ocean` array (`True` = ocean, `False` = land) -- same convention as `mask_design_wake.py`'s `ocean` array -- plus a quick render. Hand `ocean_mask.npy` back and I'll wire it (and the OBC segments) back into `wake_case.py`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ocean = topo.tmask.data.astype(bool)
np.save("ocean_mask.npy", ocean)

fig, ax = plt.subplots(figsize=(12, 8.5))
ax.imshow(ocean, origin="lower", cmap="Blues", vmin=-0.3, vmax=1.3)
ax.set_xlabel("T-column (I index)")
ax.set_ylabel("T-row (J index)")
ax.set_title("Your edited mask")
plt.tight_layout()
plt.savefig("my_topo.png", dpi=130)
print("saved ocean_mask.npy and my_topo.png")